# Baseline Solution

## Описание

В этом Jupyter-ноутбуке представлено **базовое решение** задачи бинарной классификации: **Определение релевантности изображения для карточки товара.**

Данное решение:
- Простое и быстрое в реализации, так как не требует обучения модели с нуля.
- Использует предобученную модель [**jinaai/jina-clip-v2**](https://huggingface.co/jinaai/jina-clip-v2).
- Подходит для быстрого старта, улучшения, и поможет разобраться как правильно формировать файл решения.

## Imports and constants

In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

warnings.filterwarnings("ignore")
tqdm.pandas()

Как уже было указано, в данном решении используется предобученная модель **CLIP (Contrastive Language–Image Pretraining)** от OpenAI —  а именно, её усовершенствованная версия от Jina AI:
[**jinaai/jina-clip-v2**](https://huggingface.co/jinaai/jina-clip-v2).

Решение основано на анализе семантической схожести пары **изображение — текст** (в нашем случае текст — это название товара и его описание).  

Модель Jina CLIP v2 расширяет оригинальную архитектуру CLIP, предлагая улучшенное понимание как на английском, так и на множестве других языков, включая русский, а также демонстрирует повышенную точность в задачах сопоставления изображений и текстов. Мы используем модель следующим образом:
1. Для каждой пары *изображение — название товара + его описание* вычисляется их **схожесть (similarity score)**. В качестве схожести используем **косинусную близость**.
2. Этот скор интерпретируется как **вероятность релевантности** изображения для данной карточки товара.

In [ ]:
MODEL_NAME = "jinaai/jina-clip-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Замените пути на свои при необходимости
DATA_FOLDER = Path(r"/home/vladg00dman/Projects/Hakatons/RWB_ML/ML_solve/Vlad/data")
IMAGE_FOLDER = DATA_FOLDER / "images"

print(f"Currently using {DEVICE}")

Currently using cuda


## Model Initialization

Загружаем модель с huggingface. Будем использовать библиотеку [sentence_transformers](https://sbert.net) 

In [ ]:
from sentence_transformers import SentenceTransformer

truncate_dim = 512          # или None, если хотите полные 1024-dim
model = SentenceTransformer(
    "jinaai/jina-clip-v2",
    trust_remote_code=True,
    truncate_dim=truncate_dim   # ← ЭТОГО НЕТ В ВАШЕМ БАЗОВОМ КОДЕ
)
model.to(DEVICE)
print(f"Model is currently on {model.device}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/999 [00:00<?, ?it/s]

Model is currently on cuda:0


## Data Loading

Поскольку в данном решении мы не проводим обучение модели, необходимость в полной обучающей выборке отсутствует.  

Тем не менее, для оценки качества нашего подхода мы выделим небольшую **валидационную подвыборку** из обучающих данных. С её помошью:
- Проверим корректность реализации логики инференса.
- Оценим целевую метрику качества на размеченных данных.
- Получить приблизительное представление о том, насколько хорошо модель справляется с задачей определения релевантности.

В дальнейшем эта подвыборка будет обозначаться как `val_df`

In [4]:
train_df = pd.read_csv(DATA_FOLDER / "train.csv")
test_df = pd.read_csv(DATA_FOLDER / "test.csv")

val_df = train_df.head(500)

В качестве меры схожности, как было сказано ранее, применяется **косинусная близость (cosine similarity)** между векторными представлениями (эмбеддингами) изображения и текста.

Следующий код реализует: 
- Базовую предобработку текстов (конкатенацию названия и описания)
- Вычисление поэлементного косинусного сходства для соответствующих пар

Чем ближе значение косинусной близости к 1, тем выше предполагаемая релевантность изображения для данного текстового описания.

In [5]:
def combine_texts(row):
    return f"Название товара: {row['name']}. Описание товара: {row['description']}"


batch_size = 16
similarities = []

image_paths = [os.path.join(IMAGE_FOLDER, f"{row_id}.jpg") for row_id in val_df["id"]]
texts = val_df.apply(combine_texts, axis=1).tolist()

for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
    text_batch = texts[i : i + batch_size]
    img_batch = image_paths[i : i + batch_size]

    text_embeds = model.encode(
        text_batch,
        normalize_embeddings=True,
        batch_size=len(text_batch),
        show_progress_bar=False,
    )
    image_embeds = model.encode(
        img_batch,
        normalize_embeddings=True,
        batch_size=len(img_batch),
        show_progress_bar=False,
    )

    text_embeds = torch.tensor(text_embeds).to(DEVICE)
    image_embeds = torch.tensor(image_embeds).to(DEVICE)

    cos_sims = util.cos_sim(text_embeds, image_embeds)
    diag_sims = torch.diag(cos_sims).cpu().numpy()

    similarities.extend(diag_sims)

val_df = val_df.copy()
val_df["similarity_score"] = similarities

Batches: 100%|██████████| 32/32 [00:28<00:00,  1.11it/s]


In [6]:
val_df.isna().sum()

id                      0
card_identifier_id      0
label                   0
name                    0
description             0
similarity_score      500
dtype: int64

Теперь мы можем оценить качество полученных предсказаний с помощью метрики **ROC AUC**.

ROC AUC интерпретируется как вероятность того, что случайно выбранная положительная пара (релевантное изображение) получит от модели более высокий скор, чем случайно выбранная отрицательная пара (нерелевантное изображение).  
Значение:
- **ROC AUC = 0.5** — модель не лучше случайного угадывания.
- **ROC AUC > 0.5** — модель устойчиво различает классы.
- **ROC AUC < 0.5** — модель систематически "путает" классы: присваивает более высокие скоры нерелевантным изображениям, чем релевантным.

In [7]:
roc_auc_val = roc_auc_score(val_df["label"], val_df["similarity_score"])
print(f"ROC AUC на валидации:{roc_auc_val: .3f}")

ValueError: Input contains NaN.

Аналогично получим схожесть для тестовой выборки.

In [ ]:
image_paths = [os.path.join(IMAGE_FOLDER, f"{row_id}.jpg") for row_id in test_df["id"]]
texts = test_df.apply(combine_texts, axis=1).tolist()

similarities = []

for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
    text_batch = texts[i : i + batch_size]
    img_batch = image_paths[i : i + batch_size]

    text_embeds = model.encode(
        text_batch,
        normalize_embeddings=True,
        batch_size=len(text_batch),
        show_progress_bar=False,
    )
    image_embeds = model.encode(
        img_batch,
        normalize_embeddings=True,
        batch_size=len(img_batch),
        show_progress_bar=False,
    )

    text_embeds = torch.tensor(text_embeds).to(DEVICE)
    image_embeds = torch.tensor(image_embeds).to(DEVICE)

    cos_sims = util.cos_sim(text_embeds, image_embeds)
    diag_sims = torch.diag(cos_sims).cpu().numpy()

    similarities.extend(diag_sims)

test_df["y_pred"] = similarities

Batches:   0%|          | 0/527 [00:00<?, ?it/s]

Batches: 100%|██████████| 527/527 [01:59<00:00,  4.41it/s]


## Нормализация сходства: приведение скоров к диапазону [0, 1]

Значения косинусной близости, возвращаемые моделью CLIP, лежат в диапазоне **от -1 до 1**, однако, для совместимости с **требованиями платформы соревнования**, предсказания **должны быть представлены в виде вероятностей**, то есть в диапазоне **[0, 1]**.

Это преобразование можно выполнить с помощью **сигмоидной функции**:

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$
Применение сигмоиды не повлияет на метрику так как **сохраняет порядок** предсказаний — если одно изображение было оценено как более релевантное до преобразования, оно останется более релевантным после.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


test_df["y_pred"] = sigmoid(test_df["y_pred"])

Сформируем csv файл решения. Полученный файл можно отправить на платформу.

In [ ]:
test_df[["id", "y_pred"]].to_csv(DATA_FOLDER / "sample_submission.csv", index=None)

Если весь процесс прошёл удачно, то вы можете ожидать метрику на лидерборде $\approx 0.64$ ROC AUC